In [ ]:
!pip install jax "jax[cuda13]" transformers huggingface_hub einops

# LLaDA - Large Language Diffusion with mAsking

**Paper:** [arXiv:2502.09992](https://arxiv.org/abs/2502.09992) - *LLaDA: Large Language Diffusion with mAsking* (Nie et al., 2025).  
**HF source:** [GSAI-ML/LLaDA-8B-Instruct](https://huggingface.co/GSAI-ML/LLaDA-8B-Instruct)

This notebook walks through a **from-scratch JAX implementation** of LLaDA-8B.

---

### Paper Overview

LLaDA reframes text generation as a **masked diffusion** process. Rather than predicting tokens left-to-right, a standard LLaMA transformer is trained to iteratively *unmask* corrupted token sequences — closer in spirit to BERT than GPT, but operating at generative scale.

| Innovation | What it solves |
|---|---|
| **Masked Diffusion LM** | Generates all output tokens simultaneously via iterative unmasking |
| **Block Denoising** | Splits the generation region into fixed-length blocks; each block is denoised independently |
| **Classifier-Free Guidance** | Conditions generation without a separate reward model by contrasting conditional vs. unconditional logits |
| **LLaMA backbone** | Full transformer (RoPE, RMSNorm, SwiGLU) repurposed as the denoiser network |

### Generation at a Glance

```
Input:   [prompt tokens | [MASK] [MASK] ... [MASK]]   ← gen_length mask tokens appended
              ↓
         Token Embedding  (vocab_size → d_model=4096)
              ↓
         32× LLaMA Block
              RMSNorm → RoPE Multi-Head Attention (32 heads) → residual
              RMSNorm → SwiGLU FFN (hidden=12288) → residual
              ↓
         RMSNorm + LM Head  →  logits  (B, S, vocab_size)
              ↓
         Per masked position: score confidence, unmask top-k most confident
         Repeat for num_steps per block  →  fully decoded sequence
```

### Download

Download the LLaDA-8B-Instruct weights from Hugging Face. The model is sharded across multiple `.safetensors` files (~16 GB total).

In [ ]:
HF_REPO_ID = "GSAI-ML/LLaDA-8B-Instruct"
LOCAL_DIR_PATH = "llada"

In [ ]:
import os
os.environ["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxxx" # Your HF Token here

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id=HF_REPO_ID, local_dir=f'{LOCAL_DIR_PATH}', local_dir_use_symlinks=False)

### Imports

In [ ]:
import gc
import json
import os
from dataclasses import dataclass
from functools import partial
from typing import Optional

import jax
import jax.nn as jnn
import jax.numpy as jnp
import numpy as np
from einops import einsum, rearrange, repeat
from PIL import Image
from safetensors.torch import load_file


### Load Weights

LLaDA-8B is stored as multiple `.safetensors` shards. `load_all_shards` iterates over every shard in sorted order and merges them into a single flat dictionary. We use `.pop()` during weight extraction later so that we can verify nothing was left behind.

In [ ]:
WEIGHTS_PATH = f"{LOCAL_DIR_PATH}/model.safetensors"
CONFIG_PATH = f"{LOCAL_DIR_PATH}/config.json"
TOKENIZER_PATH = LOCAL_DIR_PATH

def load_all_shards(
    directory: str,
) -> dict:
    hf_weights = {}
    shards = sorted([f for f in os.listdir(directory) if f.endswith(".safetensors")])
    for shard in shards:
        hf_weights.update(load_file(os.path.join(directory, shard), device="cpu"))
    return hf_weights

hf_weights = load_all_shards(TOKENIZER_PATH)

### Config

All model hyperparameters in one place. LLaDA-8B follows a standard LLaMA-3-style architecture: `d_model=4096`, 32 heads, 32 layers, SwiGLU FFN with hidden size 12288, and a large vocab of 126 464 tokens. The key diffusion-specific additions are `mask_token_id` (the `[MASK]` token used during generation) and `gen_length` (how many tokens to generate).

In [3]:
@dataclass
class Config:
    d_model: int = 4096
    n_heads: int = 32
    n_kv_heads: int = 32    
    n_layers: int = 32
    embedding_dropout: float = 0.0
    max_sequence_length: int = 4096
    rope_theta: float = 500000.0
    include_qkv_bias: bool = True
    include_bias: bool = True
    vocab_size: int = 126464
    embedding_size: int = 126464
    mlp_hidden_size: int = 12288
    block_group_size: int = 1
    rms_norm_eps: float = 1e-5
    attn_drop: float = 0.0
    emb_drop: float = 0.0
    resid_drop: float = 0.0
    eos_idx: int = 126081
    eot_idx: int = 126348
    mask_token_id: int = 126336
    pad_token_id: int = 126081
    gen_length: int = 64

config = Config()

### Inspect Weights

Quick sanity check — confirm the total number of weight tensors loaded from the shards.

In [4]:
len(hf_weights.keys())

291

### Device Sharding

JAX can spread large weight matrices across multiple accelerators. Here we define two sharding strategies — **row sharding** (split along axis 0) and **column sharding** (split along axis 1) — used when loading weights to distribute the 8B parameters across available devices.

In [5]:
from jax.sharding import Mesh, PartitionSpec as P, NamedSharding
from jax.experimental import mesh_utils

devices = mesh_utils.create_device_mesh((2,))
mesh = Mesh(devices, axis_names=('model',))

row_sharding = NamedSharding(mesh, P('model', None)) 
col_sharding = NamedSharding(mesh, P(None, 'model'))

### Weight Extraction

`get_w` pops a tensor from the raw HF checkpoint, converts it to `bfloat16`, optionally transposes it, and optionally places it on a specific device shard. The loop then builds the `m` dict — a clean, structured representation of all 32 transformer layers plus the embedding and LM-head weights.

Each layer has two sub-dicts:
- **`attention`** — norm, Q/K/V projections, and output projection
- **`ff`** — norm, gate projection, up projection, and down projection

In [ ]:
def get_w(
    name: str,
    transpose: bool = False,
    shard_axis: Optional[int] = None,
) -> jnp.ndarray:
    val = hf_weights.pop(name)
    w = jnp.array(val.detach().cpu(), dtype=jnp.bfloat16)
    del val

    if transpose:
        w = w.T

    if shard_axis == 0:
        current_sharding = row_sharding
    elif shard_axis == 1:
        current_sharding = col_sharding
    
    if shard_axis is not None:
        w = jax.device_put(w, current_sharding)

    gc.collect()
    return w

layers = []
for i in range(config.n_layers):
    layer_params = {
        'attention': {
            'norm': get_w(f'model.transformer.blocks.{i}.attn_norm.weight'),
            'q': get_w(f'model.transformer.blocks.{i}.q_proj.weight', transpose=True, shard_axis=1),
            'k': get_w(f'model.transformer.blocks.{i}.k_proj.weight', transpose=True, shard_axis=1),
            'v': get_w(f'model.transformer.blocks.{i}.v_proj.weight', transpose=True, shard_axis=1),
            'out': get_w(f'model.transformer.blocks.{i}.attn_out.weight', transpose=True, shard_axis=0),
        },
        'ff': {
            'norm': get_w(f'model.transformer.blocks.{i}.ff_norm.weight'),
            'gate': get_w(f'model.transformer.blocks.{i}.ff_proj.weight', transpose=True, shard_axis=1),
            'up': get_w(f'model.transformer.blocks.{i}.up_proj.weight', transpose=True, shard_axis=1),
            'out': get_w(f'model.transformer.blocks.{i}.ff_out.weight', transpose=True, shard_axis=0),
        }
    }
    layers.append(layer_params)

m = {
    'norm': get_w('model.transformer.ln_f.weight'),
    'lm_head': get_w('model.transformer.ff_out.weight', shard_axis=0),
    'embed_tokens': get_w('model.transformer.wte.weight', shard_axis=0),
    'layers': layers
}

### Verify Weight Consumption

After extraction, the `hf_weights` dict should be empty. Any remaining keys indicate weights that were not consumed — useful for catching typos in weight names.

In [7]:
hf_weights.keys()

dict_keys([])

### Rotary Position Embeddings (RoPE)

RoPE encodes position by *rotating* query and key vectors in the complex plane, so relative position is captured directly in the dot-product attention score.

**`precompute_rope_table`** — builds `sin` and `cos` tables of shape `(max_seq_len, head_dim)` once at startup.  
**`get_rope_embeddings`** — slices and reshapes the tables for the actual sequence positions.  
**`apply_rope`** — rotates each head's Q or K vector: the first half of `head_dim` is paired with the second half (the "rotate half" trick), then blended with the position-dependent `cos`/`sin`.

In [ ]:
def precompute_rope_table() -> tuple[jnp.ndarray, jnp.ndarray]:
    head_dim = config.d_model // config.n_heads
    t = jnp.arange(config.max_sequence_length, dtype=jnp.float32)
    
    inv_freq = 1.0 / (config.rope_theta ** (jnp.arange(0, head_dim, 2)[: (head_dim // 2)] / head_dim))
    freqs = jnp.outer(t, inv_freq)
    emb = jnp.concatenate((freqs, freqs), axis=-1)
    
    return jnp.sin(emb), jnp.cos(emb)


def get_rope_embeddings(
    position_ids: jnp.ndarray,  # (B, S)
    sin_table: jnp.ndarray,     # (max_seq_len, head_dim)
    cos_table: jnp.ndarray,     # (max_seq_len, head_dim)
) -> tuple[jnp.ndarray, jnp.ndarray]:
    sin = sin_table[position_ids]
    cos = cos_table[position_ids]
    sin = rearrange(sin, 'b s d -> b s 1 d')
    cos = rearrange(cos, 'b s d -> b s 1 d')
    return sin, cos

def apply_rope(x, cos, sin):
    
    half = x.shape[-1] // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    rotate_half_x = jnp.concatenate([-x2, x1], axis=-1)
    
    return x * cos + rotate_half_x * sin

sin_table, cos_table = precompute_rope_table()
print(sin_table.shape, cos_table.shape)

(4096, 128) (4096, 128)


### Diffusion Schedule — Transfer Tokens

In masked diffusion, generation is split into **blocks** of fixed length. Within each block, tokens are unmasked over `steps_per_block` steps. `get_num_transfer_tokens` computes how many tokens to unmask at each step — distributing `block_length` tokens as evenly as possible across all steps, with remainder tokens added to the early steps.

In [ ]:
def get_num_transfer_tokens(
    batch_size: int,
    block_length: int,
    steps_per_block: int,
) -> jnp.ndarray:  # (batch_size, steps_per_block)
    base_steps_per_iter = block_length // steps_per_block
    additional_steps = block_length % steps_per_block
    batch_size = 1

    return jnp.full((batch_size, steps_per_block), base_steps_per_iter) + (jnp.arange(steps_per_block) < additional_steps)

### Position IDs

Computes a position index for each token in the sequence. Tokens with `attention_mask == 1` get their true position; padding tokens get position 0. This feeds directly into `get_rope_embeddings`.

In [ ]:
def calculate_position_ids(
    attention_mask: jnp.ndarray,    # (B, S)
) -> jnp.ndarray:                   # (B, S)
    
    seq_length = attention_mask.shape[1]
    position_ids = jnp.where(attention_mask == 1, jnp.arange(seq_length), 0)
    return position_ids

### Gumbel Noise

Adding Gumbel noise to logits before taking `argmax` is equivalent to sampling from the categorical distribution — a differentiable-friendly alternative to explicit multinomial sampling. Temperature controls the sharpness: `temperature=0` gives greedy argmax, higher values add more randomness.

In [ ]:
def add_gumbel_noise(
    logits: jnp.ndarray,  # (B, S, vocab_size)
    key: jax.Array,
    temperature: float = 0.5,
) -> jnp.ndarray:         # (B, S, vocab_size)
    
    if temperature == 0:
        return logits
    u = jax.random.uniform(key, logits.shape, jnp.float32, minval=1e-6, maxval=1.0)
    gumbel = -jnp.log(-jnp.log(u))
    return logits + temperature * gumbel

### Confidence Ranking

Converts raw confidence scores into dense integer ranks (0 = least confident). Used during generation to identify the top-k most confident masked positions that should be unmasked at each step.

In [ ]:
def rank_fn(confidence):
    rank = jax.numpy.argsort(confidence, axis=-1)
    return jax.numpy.argsort(rank, axis=-1)

### RMS Normalisation

RMSNorm is a simplified layer norm that skips the mean-centering step — it only scales by the RMS of the activations. This is the normalization used throughout LLaMA-style models (pre-norm applied before each sub-layer).

In [ ]:
def rms_norm(x, weight, eps=config.rms_norm_eps):
    return x * jax.lax.rsqrt(jnp.mean(jnp.square(x), axis=-1, keepdims=True) + eps) * weight

### LLaMA Transformer Block

One full transformer layer. Follows the standard pre-norm LLaMA pattern:

```
x → RMSNorm → RoPE Multi-Head Attention → residual (+x)
  → RMSNorm → SwiGLU FFN → residual (+x)
```

**Attention** — queries and keys are rotated by RoPE before the scaled dot-product. An optional causal or padding mask is applied to the logits.

**SwiGLU FFN** — the feed-forward block uses a *gated* activation: `SiLU(gate) × up`, then projected back down. This gating mechanism is what makes SwiGLU outperform vanilla SiLU/ReLU FFNs.

In [ ]:
def llama_block(
    x: jnp.ndarray,               # (B, S, d_model)
    sin: jnp.ndarray,             # (B, S, 1, head_dim)
    cos: jnp.ndarray,             # (B, S, 1, head_dim)
    mask: Optional[jnp.ndarray],  # (B, 1, 1, S) or None
    layer_params: dict,
) -> jnp.ndarray:                 # (B, S, d_model)
    h = rms_norm(x, layer_params['attention']['norm'])
    
    q = h @ layer_params['attention']['q']
    k = h @ layer_params['attention']['k']
    v = h @ layer_params['attention']['v']
    
    q = rearrange(q, 'b s (h d) -> b s h d', h=config.n_heads)
    k = rearrange(k, 'b s (h d) -> b s h d', h=config.n_kv_heads)
    v = rearrange(v, 'b s (h d) -> b s h d', h=config.n_kv_heads)
    
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)
    
    scale = 1.0 / jnp.sqrt(config.d_model // config.n_heads)
    logits = jnp.einsum("btnh,bsnh->bnts", q, k) * scale
    
    if mask is not None:
        logits = jnp.where(mask, logits, -1e9)
        
    probs = jax.nn.softmax(logits.astype(jnp.float32), axis=-1).astype(x.dtype)
    
    attn_out = jnp.einsum("bnts,bsnh->btnh", probs, v)
    attn_out = attn_out.reshape(x.shape[0], x.shape[1], -1)
    attn_out = attn_out @ layer_params['attention']['out']
    
    x = x + attn_out
    
    res = x
    h = rms_norm(x, layer_params['ff']['norm'])
    
    gate = h @ layer_params['ff']['gate']
    up = h @ layer_params['ff']['up']
    h_mlp = jax.nn.silu(gate) * up  
    out_mlp = h_mlp @ layer_params['ff']['out']
    
    return res + out_mlp

### Full Model Forward Pass

Stacks all 32 LLaMA blocks into a single `@jax.jit`-compiled forward pass. The input token IDs are embedded, position IDs are computed from the attention mask, RoPE sin/cos tables are sliced, and the sequence passes through every layer. The final RMSNorm + `lm_head` projection produces logits over the full vocabulary for every position.

In [ ]:
@jax.jit
def model(
    x: jnp.ndarray,               # (B, S)
    attention_mask: jnp.ndarray,  # (B, S)
    params: dict,
) -> jnp.ndarray:                 # (B, S, vocab_size)
    x = params['embed_tokens'][x]
    position_ids = calculate_position_ids(attention_mask)
    sin, cos = get_rope_embeddings(position_ids, sin_table, cos_table)

    if attention_mask is not None:
        attention_mask = rearrange(attention_mask, 'b s -> b 1 1 s')
    
    for layer_params in params['layers']:
        x = llama_block(x, sin, cos, attention_mask, layer_params)
        
    x = rms_norm(x, params['norm'])
    logits = x @ params['lm_head'].T
    
    return logits

### Block-Level Generation (Basic)

An early generation implementation using `jax.lax.scan` for XLA-efficient looping within each block. At every step:
1. Run the full model forward pass to get logits
2. Sample a candidate token `x0` for every position via Gumbel noise
3. Score each masked position by its model confidence (max prob)
4. Unmask the top-`k` most confident positions (where `k = step_transfer_tokens[step]`)

The outer `generate` function appends `gen_length` mask tokens to the prompt and iterates over blocks.

In [ ]:
@jax.jit
def generate_block(
    params: dict,
    x: jnp.ndarray,                       # (B, S)
    attention_mask: jnp.ndarray,          # (B, S)
    key: jax.Array,
    step_transfer_tokens: jnp.ndarray,    # (num_steps,)
) -> tuple[jnp.ndarray, jax.Array]:
    def scan_body(carry, step_tokens):
        current_x, current_key = carry
        
        logits = model(current_x, attention_mask, params)
        
        current_key, subkey = jax.random.split(current_key)
        noise = jax.random.gumbel(subkey, logits.shape)
        x0 = jnp.argmax(logits + noise * 0.2, axis=-1)
        
        probs = jax.nn.softmax(logits.astype(jnp.float32), axis=-1)
        x0_p = jnp.max(probs, axis=-1)
        
        mask_index = (current_x == config.mask_token_id)
        confidence = jnp.where(mask_index, x0_p, -1.0)

        ranks = jnp.argsort(jnp.argsort(confidence, axis=-1), axis=-1)
        threshold = confidence.shape[-1] - step_tokens
        topk_mask = ranks >= threshold
        
        next_x = jnp.where(topk_mask, x0, current_x)
        
        return (next_x, current_key), None

    (final_x, final_key), _ = jax.lax.scan(scan_body, (x, key), step_transfer_tokens)
    return final_x, final_key

def generate(
    input_ids: jnp.ndarray,  # (B, prompt_len)
    params: dict,
) -> jnp.ndarray:            # (B, prompt_len + gen_length)
    key = jax.random.PRNGKey(42)
    
    total_seq = jnp.concatenate([
        input_ids, 
        jnp.full((1, config.gen_length), config.mask_token_id, dtype=jnp.int32)
    ], axis=1)
    
    attn_mask = jnp.ones(total_seq.shape, dtype=jnp.int32)

    block_length = 32
    num_blocks = config.gen_length // block_length
    num_steps = 32

    block_schedule = get_num_transfer_tokens(1, block_length, num_steps)[0]

    x = total_seq
    for b in range(num_blocks):
        print(f"Running Block {b}...")
        x, key = generate_block(params, x, attn_mask, key, block_schedule)
    return x

### Classifier-Free Guidance (CFG) Generation

An enhanced generation strategy. CFG runs two model forward passes per step — one **conditional** (on the real prompt) and one **unconditional** (where the entire prompt is replaced with `[MASK]` tokens) — then linearly extrapolates away from the unconditional prediction:

```
logits = logits_cond + cfg_scale × (logits_cond − logits_uncond)
```

A higher `cfg_scale` (here 9.0) pushes the model harder towards the conditional distribution, improving instruction-following at the cost of 2× compute per step. The rest of the unmasking logic (Gumbel sampling, confidence ranking, top-k reveal) is identical to the basic version.

In [ ]:
@partial(jax.jit, static_argnums=(5,))
def generate_block_cfg(
    params: dict,
    x: jnp.ndarray,               # (B, S)
    attention_mask: jnp.ndarray,  # (B, S)
    key: jax.Array,
    block_schedule: jnp.ndarray,  # (num_steps,)
    prompt_len: int,
) -> tuple[jnp.ndarray, jax.Array]:
    
    cfg_scale = 9.0
    
    def scan_body(carry, step_tokens):
        current_x, current_key = carry
        uncond_x = jnp.full_like(current_x, config.mask_token_id)
        
        logits_cond = model(current_x, attention_mask, params)
        logits_uncond = model(uncond_x, attention_mask, params)
        logits = logits_cond + cfg_scale * (logits_cond - logits_uncond)
        
        current_key, subkey = jax.random.split(current_key)
        logits = add_gumbel_noise(logits, subkey, temperature=0.2)
        
        x0 = jnp.argmax(logits, axis=-1)
        probs = jax.nn.softmax(logits.astype(jnp.float32), axis=-1)
        x0_probs = jnp.max(probs, axis=-1)
        
        mask_index = (current_x == config.mask_token_id)
        confidence = jnp.where(mask_index, x0_probs, -1.0)
        ranks = jnp.argsort(jnp.argsort(confidence, axis=-1), axis=-1)
        
        threshold = confidence.shape[-1] - step_tokens
        topk_mask = ranks >= threshold
        next_x = jnp.where(topk_mask, x0, current_x)
        
        return (next_x, current_key), None

    (final_x, final_key), _ = jax.lax.scan(scan_body, (x, key), block_schedule)
    return final_x, final_key

def generate(
    input_ids: jnp.ndarray,  # (B, prompt_len)
    params: dict,
) -> jnp.ndarray:            # (B, prompt_len + gen_length)
    key = jax.random.PRNGKey(42)
    
    prompt_len = input_ids.shape[1]
    total_seq = jnp.concatenate([
        input_ids, 
        jnp.full((1, config.gen_length), config.mask_token_id, dtype=jnp.int32)
    ], axis=1)
    
    attn_mask = jnp.ones(total_seq.shape, dtype=jnp.int32)
    
    block_length = 32
    num_blocks = config.gen_length // block_length
    
    num_steps = 32
    block_schedule = get_num_transfer_tokens(1, block_length, num_steps)[0]
    x = total_seq
    
    for b in range(num_blocks):
        print(f"Running Block {b}...")
        x, key = generate_block_cfg(params, x, attn_mask, key, block_schedule, prompt_len)
    
    return x

### Generation Loop (Step-by-Step)

The final `generate` used at inference. Rather than jitting the inner loop with `lax.scan`, this version runs in plain Python — useful for printing per-step and per-block progress, experimenting with block sizes, and debugging the unmasking schedule. CFG is stubbed out here (commented in the cell) so you can toggle it on/off without recompiling.

Block-level masking (`confidence.at[:, :block_start].set(-inf)`) ensures only positions inside the *current* block are candidates for unmasking at each step.

In [ ]:
def generate(
    input_ids: jnp.ndarray,  # (B, prompt_len)
    params: dict,
) -> jnp.ndarray:            # (B, prompt_len + gen_length)
    
    prompt_len = input_ids.shape[1]
    x = jnp.concatenate([
        input_ids,
        jnp.full((1, config.gen_length), config.mask_token_id, dtype=jnp.int32)
    ], axis=1)
    key = jax.random.PRNGKey(42)

    attention_mask = jnp.ones(x.shape, dtype=jnp.int32)

    block_length = 64
    num_blocks = config.gen_length // block_length
    steps_per_block = 32

    cfg_scale = 9.0

    for block in range(num_blocks):
        print(f"Running Block {block}...")
        block_start = prompt_len + block * block_length
        block_end   = prompt_len + (block + 1) * block_length
        num_transfer_tokens = get_num_transfer_tokens(x.shape[0], block_length, steps_per_block)

        for step in range(steps_per_block):
            print(f"Running Step {step}...")

            logits_cond = model(x, attention_mask, params)

            # uncond_x = x.at[:, :prompt_len].set(config.mask_token_id)
            
            # logits_uncond = model(uncond_x, attention_mask, params)
            # logits = logits_uncond + cfg_scale * (logits_cond - logits_uncond)
            logits = logits_cond

            key, subkey = jax.random.split(key)
            # logits_noised = add_gumbel_noise(logits, subkey, temperature=1.0)
            logits_noised = logits
            x0 = jnp.argmax(logits_noised, axis=-1)

            probs = jax.nn.softmax(logits_noised.astype(jnp.float32), axis=-1)
            x0_p = jnp.max(probs, axis=-1)

            mask_index = (x == config.mask_token_id)
            confidence = jnp.where(mask_index, x0_p, -jnp.inf)

            confidence = confidence.at[:, :block_start].set(-jnp.inf)
            confidence = confidence.at[:, block_end:].set(-jnp.inf)

            ranks = rank_fn(confidence)
            threshold = (confidence.shape[-1] - num_transfer_tokens[:, step])[:, None]
            topk_mask = ranks >= threshold

            x = jnp.where(topk_mask, x0, x)

    return x

### Tokenise & Run

Load the LLaDA tokenizer and format the prompt using the instruct chat template. The result is a NumPy array of token IDs that gets passed directly to `generate`.

In [91]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, trust_remote_code=True)
prompt = [
    {"role": "user", "content": "Name some cities in France."}
]

input_ids = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, return_tensors="np")['input_ids']
# input_ids = tokenizer("Name some cities in France", return_tensors="np")['input_ids']

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


In [ ]:
jax.clear_caches()
gc.collect()
result = generate(input_ids, m)

Running Block 0...
Running Step 0...


### Decode Output

Convert the generated token IDs back into a human-readable string using the tokenizer.

In [ ]:
tokenizer.decode(result[0])

### Visualisation — Unmasking Progression

This is the core LLaDA idea made visible. We re-run the generation loop on a short prompt with a small `gen_length` and capture the full token sequence after every denoising step.

Each row is one step. Each column is a token position in the generation region:
- **Dark** — still masked `[M]`
- **Teal** — just revealed this step
- **Green** — already revealed in a previous step
- **Grey** — prompt (fixed, never masked)

You'll see the model first commit to its most confident positions, then progressively fill in the rest.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

vis_prompt = [{"role": "user", "content": "Name some cities in France."}]
vis_input_ids = jnp.array(
    tokenizer.apply_chat_template(vis_prompt, add_generation_prompt=True, return_tensors="np")['input_ids'],
    dtype=jnp.int32
)

VIS_GEN_LENGTH = 24
VIS_STEPS      = 12
prompt_len     = vis_input_ids.shape[1]

x_vis = jnp.concatenate([vis_input_ids, jnp.full((1, VIS_GEN_LENGTH), config.mask_token_id, dtype=jnp.int32)], axis=1)
attn_vis = jnp.ones(x_vis.shape, dtype=jnp.int32)
ntt = get_num_transfer_tokens(1, VIS_GEN_LENGTH, VIS_STEPS)

gen_snapshots   = [np.array(x_vis[0, prompt_len:])]     # only the generation region
step_newly      = [set()]                               # which positions were revealed each step

for step in range(VIS_STEPS):
    logits = model(x_vis, attn_vis, m)
    probs  = jax.nn.softmax(logits.astype(jnp.float32), axis=-1)
    x0     = jnp.argmax(logits, axis=-1)
    x0_p   = jnp.max(probs, axis=-1)

    mask_index = (x_vis == config.mask_token_id)
    confidence = jnp.where(mask_index, x0_p, -jnp.inf)
    ranks      = rank_fn(confidence)
    threshold  = (confidence.shape[-1] - ntt[:, step])[:, None]
    topk_mask  = ranks >= threshold

    prev_gen = np.array(x_vis[0, prompt_len:])
    x_vis    = jnp.where(topk_mask, x0, x_vis)
    curr_gen = np.array(x_vis[0, prompt_len:])

    newly = {i for i in range(VIS_GEN_LENGTH) if prev_gen[i] == config.mask_token_id and curr_gen[i] != config.mask_token_id}
    gen_snapshots.append(curr_gen)
    step_newly.append(newly)

all_ids = {int(t) for snap in gen_snapshots for t in snap}
id_to_str = {}
for tid in all_ids:
    raw = tokenizer.decode([tid], skip_special_tokens=False)
    raw = raw.replace('\n', '↵').replace(' ', '·').strip() or '?'
    id_to_str[tid] = raw[:5]

n_rows = len(gen_snapshots)
n_cols = VIS_GEN_LENGTH
cell_w, cell_h = 1.1, 0.72

fig, ax = plt.subplots(figsize=(n_cols * cell_w * 0.55, n_rows * cell_h))
fig.patch.set_facecolor('#1e1e2e')
ax.set_facecolor('#1e1e2e')
ax.set_xlim(0, n_cols)
ax.set_ylim(0, n_rows)
ax.invert_yaxis()
ax.axis('off')

for row_idx, snap in enumerate(gen_snapshots):
    newly = step_newly[row_idx]
    for col_idx, tok_id in enumerate(snap):
        tok_id = int(tok_id)
        is_mask = tok_id == config.mask_token_id

        if is_mask:
            bg, fg, label = '#181825', '#45475a', '[M]'
        elif col_idx in newly:
            bg, fg, label = '#89dceb', '#1e1e2e', id_to_str[tok_id]
        else:
            bg, fg, label = '#a6e3a1', '#1e1e2e', id_to_str[tok_id]

        rect = plt.Rectangle((col_idx + 0.05, row_idx + 0.05), 0.88, 0.82,
                              facecolor=bg, edgecolor='#313244', linewidth=0.4)
        ax.add_patch(rect)
        ax.text(col_idx + 0.49, row_idx + 0.46, label,
                ha='center', va='center', fontsize=5.5,
                color=fg, fontweight='bold' if not is_mask else 'normal',
                fontfamily='monospace')

for i, snap in enumerate(gen_snapshots):
    row_label = 'init' if i == 0 else f's{i}'
    n_revealed = int(np.sum(snap != config.mask_token_id))
    ax.text(-0.2, i + 0.46, row_label, ha='right', va='center', fontsize=6.5, color='#cdd6f4')
    ax.text(n_cols + 0.15, i + 0.46, f'{n_revealed}/{VIS_GEN_LENGTH}',
            ha='left', va='center', fontsize=6, color='#6c7086')

legend_patches = [
    mpatches.Patch(facecolor='#181825', edgecolor='#45475a', label='masked [M]'),
    mpatches.Patch(facecolor='#89dceb', edgecolor='none', label='revealed this step'),
    mpatches.Patch(facecolor='#a6e3a1', edgecolor='none', label='already revealed'),
]
ax.legend(handles=legend_patches, loc='lower center', bbox_to_anchor=(0.5, -0.06),
          ncol=3, framealpha=0, labelcolor='#cdd6f4', fontsize=8)

ax.set_title('Unmasking Progression — LLaDA Generation (generation region only)',
             color='#cdd6f4', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

### A Closer Look at the Diffusion Mechanism

The generation cells above hide the inner loop inside `jax.lax.scan` for speed. Here we unroll a **single denoising step by hand** — using dummy logits instead of running the 8B model — so we can inspect every intermediate value at a human-readable scale.

The full unmasking pipeline is just four operations:

| Step | What happens |
|---|---|
| **1. Parameters** | Define block size, number of steps, and how many tokens to reveal per step |
| **2. Build & sample** | Construct `[prompt \| MASK…]`, apply Gumbel noise → candidate tokens `x0` |
| **3. Score confidence** | Each masked position gets a confidence score; non-masked and out-of-block positions get `-inf` |
| **4. Rank & unmask** | Reveal the top-k most confident positions; write `x0` into those slots |

Run all four cells in order to trace one complete step.

**Step 1 — Define parameters**

`get_num_transfer_tokens` returns a `(1, num_steps)` schedule array — how many tokens to unmask at each step. With `block_length=64` evenly divided over 16 steps, this is 4 tokens per step.

In [ ]:
input_ids = jnp.array([[1, 2, 3, 4, 5]], dtype=jnp.int32)
prompt_len = input_ids.shape[1]

block        = 0
block_length = 64
num_steps    = 16

num_transfer_tokens = get_num_transfer_tokens(1, block_length, num_steps)
schedule = num_transfer_tokens[0]
print(f"steps per block: {num_steps}")

print(f"gen length     : {config.gen_length} tokens  ({block_length} per block × {config.gen_length // block_length} block(s))")
print(f"prompt length  : {prompt_len} tokens")

**Step 2 — Build the sequence & sample candidate tokens**

We concatenate `[prompt | MASK…]` and pad to `max_sequence_length`. Then we create **dummy logits** (random uniform — standing in for a real forward pass) and run them through `add_gumbel_noise + argmax` to get a candidate token `x0` for every position. In real generation the logits come from `model(x, attn_mask, m)`.

In [ ]:
x = jnp.concatenate([input_ids, jnp.full((1, config.gen_length), config.mask_token_id, dtype=jnp.int32)], axis=1)
x = jnp.concatenate([x, jnp.full((1, config.max_sequence_length - x.shape[1]), config.pad_token_id, dtype=jnp.int32)], axis=1)

print(f"sequence shape : {x.shape}  ({prompt_len} prompt | {config.gen_length} masked | {config.max_sequence_length - prompt_len - config.gen_length} pad)")
print(f"masked tokens  : {int(jnp.sum(x == config.mask_token_id))}")

key = jax.random.PRNGKey(0)
key, noise_key = jax.random.split(key)
step = 0


dummy_logits = jax.random.uniform(noise_key, (1, config.max_sequence_length, config.vocab_size), dtype=jnp.float32)

logits_with_noise = add_gumbel_noise(dummy_logits, noise_key, temperature=0.5)
x0 = jnp.argmax(logits_with_noise, axis=-1)

**Step 3 — Score confidence per position**

`x0_p` holds a confidence score for every position. Here we use random uniform scores as a stand-in for real model probabilities (`max(softmax(logits))`). Two positions are forced to `-inf` so they can never be selected:
- Positions **outside the current block** — only the active block can be unmasked
- Positions that are **not masked** — already-revealed tokens stay fixed

In [ ]:
key, subkey = jax.random.split(key)
x0_p = jax.random.uniform(subkey, x0.shape[:2])   # dummy stand-in for max(softmax(logits))

block_start = prompt_len + block * block_length
current_block_end = prompt_len + (block + 1) * block_length

print(f"eligible slots : {int(jnp.sum(jnp.isfinite(x0_p)))}  (non -inf positions)")

x0_p = x0_p.at[:, :block_start].set(-jnp.inf)
x0_p = x0_p.at[:, current_block_end:].set(-jnp.inf)

In [ ]:
mask_index = (x == config.mask_token_id)
confidence = jnp.where(mask_index, x0_p, -jnp.inf)  # already-revealed tokens → -inf

print(f"will unmask this step     : {int(schedule[step])} token(s)")
print(f"masked positions in block : {int(jnp.sum(mask_index[0, block_start:current_block_end]))} / {block_length}")

**Step 4 — Rank & unmask**

`rank_fn` converts raw scores into dense integer ranks (0 = least confident). We compute a threshold so that only the **top-k** ranked positions pass — where `k = num_transfer_tokens[step]`. Positions above the threshold get their candidate token written in; all others stay as-is.

In [ ]:
ranks     = rank_fn(confidence)
threshold = (confidence.shape[-1] - schedule[step])
topk_mask = ranks >= threshold

x_new = jnp.where(topk_mask, x0, x)
x = x_new

revealed = int(jnp.sum((x_new != config.mask_token_id) & (x == config.mask_token_id)))
print(f"remaining masked             : {int(jnp.sum(x_new == config.mask_token_id))}")
print(f"positions unmasked this step : {revealed}")